In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

# 官方示例

LangChain 教程：[如何使用 MultiQueryRetriever](https://python.langchain.com/v0.2/docs/how_to/MultiQueryRetriever)

API ：[MultiQueryRetriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html)

In [2]:
# Build a sample vectorDB
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load blog post
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
data = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splits = text_splitter.split_documents(data)

# VectorDB
embedding = OpenAIEmbeddings()
vectordb = Chroma.from_documents(documents=splits[:12], embedding=embedding)

## 简单使用

In [3]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

question = "What are the approaches to Task Decomposition?"
llm = ChatOpenAI(temperature=0)
retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=vectordb.as_retriever(), llm=llm
)

In [4]:
# Set logging for the queries
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [5]:
unique_docs = retriever_from_llm.invoke(question)
len(unique_docs)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. How can Task Decomposition be achieved through different methods?', '2. What strategies are commonly used for Task Decomposition?', '3. What are the various techniques for breaking down tasks in Task Decomposition?']


4

INFO:backoff:Backing off send_request(...) for 0.8s (requests.exceptions.ConnectionError: ('Connection aborted.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)))


In [6]:
unique_docs

[Document(page_content='Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.', metadata={'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:', 'language': 'en', 'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log"}),
 Document(page_content='Fig. 1. Overview of a LLM-powered auto

In [7]:
for doc in unique_docs:
    print(doc.page_content)
    print("="*30)

Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Fig. 1. Overview of a LLM-powered autonomous agent system.
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Planning

Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.
Reflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.


Memory
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” t

## 使用自己的提示

In [9]:
from typing import List

from langchain_core.output_parsers import BaseOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field


# Output parser will split the LLM result into a list of queries
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Output parser for a list of lines."""

    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return lines


output_parser = LineListOutputParser()

QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""你是一个人工智能语言模型助手。
    你的任务是生成5个不同版本的用户问题，以从向量数据库中检索相关文档。
    通过对用户问题生成多个视角，您的目标是帮助用户克服基于距离的相似性搜索的一些限制。
    以换行符分隔开这些备选问题。
    原始问题：{question}""",
)
llm = ChatOpenAI(temperature=0)

# Chain
llm_chain = QUERY_PROMPT | llm | output_parser

# Other inputs
question = "任务分解的方法有哪些？"

In [10]:
llm_chain.invoke({"question": question})

['1. 任务分解的策略有哪些可供选择？',
 '2. 有哪些方法可以用来拆分任务？',
 '3. 任务分解的技术有哪些不同的方式？',
 '4. 如何有效地将任务分解为更小的子任务？',
 '5. 有哪些途径可以帮助我将任务分解为可管理的部分？']

In [11]:
# Run
retriever = MultiQueryRetriever(
    retriever=vectordb.as_retriever(), llm_chain=llm_chain, parser_key="lines"
)  # "lines" is the key (attribute name) of the parsed output

# Results
unique_docs = retriever.invoke(question)
len(unique_docs)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. 任务分解的技术有哪些？', '2. 有哪些方法可以用来分解任务？', '3. 任务分解的策略有哪些选择？', '4. 任务分解的方式有哪些不同的选项？', '5. 有哪些途径可以将任务分解为更小的部分？']


5

In [12]:
unique_docs

[Document(page_content='Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.', metadata={'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:', 'language': 'en', 'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log"}),
 Document(page_content='Fig. 1. Overview of a LLM-powered auto

In [13]:
for doc in unique_docs:
    print(doc.page_content)
    print("="*30)

Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Fig. 1. Overview of a LLM-powered autonomous agent system.
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Planning

Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.
Reflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.


Memory
language. Essentially, the planning step is outsourced to an external tool, assuming the availability of domain-specific PDDL and a suitable planner which is common in certain robot

# 优化Naive-RAG

## 索引

In [14]:
from dotenv import load_dotenv
from langchain import hub
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载环境变量
load_dotenv()

# Load
file_path = "./data/translated_article.txt"
loader = TextLoader(file_path, encoding='utf-8')
docs = loader.load()

# Split
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)
markdown_text = "\n\n".join([doc.page_content for doc in docs])

md_header_splits = markdown_splitter.split_text(markdown_text)

chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(md_header_splits)

# Embed
embedding = OpenAIEmbeddings()

# Store
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

# Retrieve
retriever = vectorstore.as_retriever()

## 生成多个问题

In [15]:
from typing import List

from langchain_core.output_parsers import BaseOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field


# 输出解析器会将 LLM 结果拆分为查询列表
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Output parser for a list of lines."""

    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return lines


output_parser = LineListOutputParser()

QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""你是一个人工智能语言模型助手。
    你的任务是生成5个不同版本的用户问题，以从向量数据库中检索相关文档。
    通过对用户问题生成多个视角，您的目标是帮助用户克服基于距离的相似性搜索的一些限制。
    以换行符分隔开这些备选问题。
    原始问题：{question}""",
)
llm = ChatOpenAI(temperature=0)

llm_chain = QUERY_PROMPT | llm | output_parser

In [16]:
question = "什么是few shot？"

In [17]:
llm_chain.invoke({"question": question})

['1. 请解释一下什么是few shot学习？',
 '2. 能否详细介绍一下few shot的概念？',
 '3. 你能帮我理解一下few shot是什么意思吗？',
 '4. 对于few shot这个术语，你能提供更多的信息吗？',
 '5. 我对few shot的含义还不太清楚，你可以帮我解释一下吗？']

## 构建新的检索器——MultiQueryRetriever

In [18]:
from langchain.retrievers.multi_query import MultiQueryRetriever

# Run
new_retriever = MultiQueryRetriever(
    retriever=retriever, llm_chain=llm_chain, parser_key="lines"
)  # “lines”是解析输出的键（属性名称）

In [19]:
# Results
unique_docs = new_retriever.invoke(question)
len(unique_docs)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. 请解释一下什么是few shot学习？', '2. 能否详细介绍一下few shot学习的概念？', '3. 你能帮我理解一下few shot学习的含义吗？', '4. 请解释一下在机器学习中few shot是指什么？', '5. 能否简要说明一下few shot在人工智能领域中的应用？']


5

In [20]:
unique_docs

[Document(page_content='# 基础提示  \n零样本（Zero-shot）和少样本（Few-shot）学习是两种基本的模型提示方法，这两种方法最早由多篇大语言模型论文提出，并广泛用于评估大语言模型的性能。', metadata={'Header 1': '基础提示'}),
 Document(page_content='## 少样本  \n**少样本学习**通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，帮助模型更好地理解人类的意图和评价标准。因此，少样本学习通常比零样本学习有更好的表现。但这种方法会消耗更多的Token，并且当输入和输出文本较长时可能触及上下文长度的限制。  \n```\n文本: （劳伦斯在舞台上）到处跳跃，跳舞，跑步，出汗，擦脸，充分展现了他初次成名时的惊人才华。\n情感: 正面\n\n文本: 尽管所有迹象都表明相反，这部糟糕的电影还是设法冒充了一部正式的商业影片，收取全价门票，在电视上大肆宣传，自称能够娱乐小孩和成人。\n情感: 负面\n\n文本: 多年来第一次，德尼罗在情感上有了深刻的挖掘，可能是因为受到了合作伙伴出色表现的启发。\n情感: 正面', metadata={'Header 1': '基础提示', 'Header 2': '少样本'}),
 Document(page_content='## 零样本  \n**零样本学习**指的是直接将任务文本输入模型并询问结果。  \n（所有情感分析示例均来源于SST-2）  \n```\n文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```', metadata={'Header 1': '基础提示', 'Header 2': '零样本'}),
 Document(page_content='文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```  \n许多研究尝试了如何构建上下文示例以最大化性能，并发现**提示格式、训练示例及其顺序的选择可以极大地影响模型的性能**，从几乎是随机猜测到接近最优状态的表现。  \n[Zhao等人（2021年）](https://arxiv.org/abs/2102.09690)研究了少样本分类，并发现几种与LLM（他们使用的是GPT-3）相关的偏见导致了性能的高变异性：（1）*多数标签偏见*，如果示例中的

## 生成最终输出

In [21]:
# Generate
# prompt = hub.pull("rlm/rag-prompt")
system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI()
output_parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": new_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

question = "什么是few shot？"
print(f"Q: {question}\nA: ")

result = rag_chain.invoke(question)
print(result)

Q: 什么是few shot？
A: 


INFO:langchain.retrievers.multi_query:Generated queries: ['1. 请解释一下什么是few shot学习？', '2. 你能详细介绍一下few shot是什么意思吗？', '3. 为什么要学习few shot技术？它有什么特点？', '4. 请简要说明一下few shot学习的概念和应用。', '5. 在机器学习领域中，few shot是指什么？']


少样本学习通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，帮助模型更好地理解人类的意图和评价标准。这有助于模型在新任务上做出准确的预测，通常比零样本学习表现更好。
